In [ ]:
!pip install pandas
!pip install spacy
!pip install mord

In [ ]:
!pip install scipy

In [ ]:
import fasttext.util
import numpy as np
import pandas as pd
import re
import os
from scipy import stats
from scipy.stats import skew, kurtosis
from collections import Counter
import nltk
nltk.download('stopwords')
from nltk.corpus import stopwords
import spacy
import matplotlib.pyplot as plt
import plotly.graph_objects as go
import seaborn as sns
from sklearn.linear_model import Lasso
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import LeaveOneOut
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score, mean_absolute_error, precision_score, recall_score
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
import mord

ft = fasttext.load_model(r"Data/Models/cc.de.300.bin") #fasttext model path
print("Model loaded successfully.")

In [ ]:
# folder path
folder_path = r"Data/Corpus"
#output file for storing the features
output_excel_file = r"Result/Filename.xlsx"

nlp = spacy.load("de_core_news_md")

nltk.download('stopwords', quiet=True)
german_stop_words = set(stopwords.words('german'))

key_distinction_1 = ['forms_of_representation_for_key_distinction_1']
key_distinction_2 = ['forms_of_representation_for_key_distinction_2']

"""
#Example

#for adjectives
key_distinction_1 = ['naturverbunden']
key_distinction_2 = ['zivilisiert']

#average of synonyms
key_distinction_1 = ['naturverbunden', 'naturbelassen', 'rein', 'unverfälscht', 'lebendig', 'natürlich']
key_distinction_2 = ['zivilisiert', 'kultiviert', 'künstlich', 'artifiziell', 'hochentwickelt', 'gebildet', 'gekünstelt']

#nouns
key_distinction_1 = ['Natur']
key_distinction_2 = ['Kultur']
"""

all_seed_words = set(key_distinction_1 + key_distinction_2)

print("Checking OOV words for the selected synonyms:")
for w in key_distinction_1 + key_distinction_2:
    if ft.get_word_id(w) == -1:
        print(f"'{w}' is OOV!")

#find axis vector and normalize it
def get_axis_vector():
    key_distinction_1_vecs = [ft.get_word_vector(w) for w in key_distinction_1]
    key_distinction_2_vecs = [ft.get_word_vector(w) for w in key_distinction_2]
    key_distinction_1_mean = np.mean(key_distinction_1_vecs, axis=0)
    key_distinction_2_mean = np.mean(key_distinction_2_vecs, axis=0)
    axis = key_distinction_2_mean - key_distinction_1_mean
    return axis / np.linalg.norm(axis)

#find cosine similarities
def cosine_to_axis(word, axis_vec):
    v = ft.get_word_vector(word)
    norm = np.linalg.norm(v)
    if norm == 0:
        return 0.0
    v_norm = v / norm
    return float(np.dot(v_norm, axis_vec))

#handle empty lists to avoid errors
def summary_stats(scores):
    if not scores: 
        return {k: 0.0 for k in ["min", "max", "diff", "mean", "median", "std", "skew", "kurt", "q1", "q2", "q3"]}
        
    scores = np.array(scores)
    return {
        "min":    float(np.min(scores)),
        "max":    float(np.max(scores)),
        "diff":   float(np.max(scores)-np.min(scores)),
        "mean":   float(np.mean(scores)),
        "median": float(np.median(scores)),
        "std":    float(np.std(scores, ddof=1)) if len(scores) > 1 else 0.0,
        "skew":   float(stats.skew(scores, bias=False)) if len(scores) > 1 else 0.0,
        "kurt":   float(stats.kurtosis(scores, fisher=True, bias=False)) if len(scores) > 1 else 0.0,
        "q1": float(np.quantile(scores, 0.25)),
        "q2": float(np.quantile(scores, 0.50)),
        "q3": float(np.quantile(scores, 0.75)),
    }

def calculate_file_bias(text, axis_vec, seed_words):
    words = re.sub(r"[^\w\s]", "", text.lower()).split()
    
    unique_text_words = set(words)
    
    #Keeps only words that are stopwords
    neutral_candidates = unique_text_words.intersection(german_stop_words)
    
    scores = []
    for w in neutral_candidates:
        # Checks for seed words
        if w in seed_words: continue
        
        #Checks if the word exists in the FastText model
        if ft.get_word_id(w) != -1:
            scores.append(cosine_to_axis(w, axis_vec))

    if not scores: return 0.0
    return np.mean(scores)

#function for the pipline
def process_text_pipeline(text):
    #Runs SpaCy on RAW text, Filters NOUN/ADJ, Lowercases and cleans punctuation, Removes Stopwords, Checks if word exists in FastText Vocab
    doc = nlp(text)
    valid_words = []
    total_count = 0
    oov = 0
    
    for token in doc:
        #POS Tagging
        if token.pos_ in {"NOUN", "ADJ"}:
            #Lowercase & Clean
            w_clean = re.sub(r"[^\w\s]", "", token.text.lower())
            #Check Stopwords & Empty strings
            if w_clean and w_clean not in german_stop_words:
                total_count +=1 #keeps all the words
                if ft.get_word_id(w_clean) != -1: #Check FastText Vocabulary
                    valid_words.append(w_clean)
                else:
                    oov +=1

    if total_count > 0:
        pct = (oov / total_count) * 100
        print(f"Vocabulary Info: {total_count} total words, {oov} OOV ({pct:.1f}% dropped)")
    else:
        print(f"Vocabulary Info: No candidates found.")
    return valid_words

axis_vec = get_axis_vector()

text_files = sorted([f for f in os.listdir(folder_path) if f.lower().endswith(".txt")])

#standard deviation calculation from all the text documents
all_std_devs = []
all_document_data = {}
for fname in text_files:
    file_path = os.path.join(folder_path, fname)
    try:
        with open(file_path, 'r', encoding='utf-8') as f:
            text_document = f.read()

        file_bias = calculate_file_bias(text_document, axis_vec, all_seed_words)
        
        #Checks if the text is long and needs to be chunked
        if len(text_document) > nlp.max_length:
            print(f"The text document is very long ({len(text_document)} chars) and is processing in chunks.")
            content_words = []
            
            #Processes the text in chunks of 500,000 characters
            chunk_size = 500000 
            for i in range(0, len(text_document), chunk_size):
                chunk_text = text_document[i:i + chunk_size]
                content_words.extend(process_text_pipeline(chunk_text))
        else:
            content_words = process_text_pipeline(text_document)
        
        if content_words:
            scores = [cosine_to_axis(w, axis_vec) - file_bias for w in content_words]
            all_document_data[fname] = {'scores': scores}
            all_std_devs.append(np.std(scores))
    except Exception as e:
        print(f"Skipping {fname} due to error: {e}")

if not all_std_devs:
    GLOBAL_THRESHOLD = 0.1
    print("Warning: Could not calculate global threshold, using default 0.1")
else:
    average_std_dev = np.mean(all_std_devs)
    GLOBAL_THRESHOLD = average_std_dev / 2.0

print(f"\n Global Threshold found: {GLOBAL_THRESHOLD:.4f}\n")

#for storing the results
all_document_results = []
#for storing the stripe information
stripe_data = []

#stripes ranges from -0.5 to +0.5
bins = np.linspace(-0.5, 0.5, 41)
labels = [f"Stripe_{i+1} [{bins[i]:.2f}, {bins[i+1]:.2f})" for i in range(len(bins)-1)]

print(f"\nProcessing {len(text_files)} files from the folder.")

for fname, data in all_document_data.items():
    print(f" \n Analyzing: {fname}")
    
    scores = data['scores']
    scores_arr = np.array(scores)
    total_words = len(scores)

    #Calculates the statistical measures
    base_statistics = summary_stats(scores)
    abs_mean = float(np.mean(np.abs(scores_arr)))
    coverage = np.sum(np.abs(scores_arr) > 0.1) / total_words if total_words > 0 else 0.0
    
    #Calculates the Threshold Percentage
    words_removed = np.sum(np.abs(scores_arr) < GLOBAL_THRESHOLD)
    percent_removed = (words_removed / total_words) * 100 if total_words > 0 else 0.0

    #Calculates the stripe percentages
    score_series = pd.Series(scores)
    binned_scores = pd.cut(score_series, bins=bins, labels=labels, include_lowest=True)
    bin_counts = binned_scores.value_counts()
    bin_percentages = (bin_counts / total_words) * 100
    stripes_data = {label: bin_percentages.get(label, 0.0) for label in labels}

    #Combines all results into a single dictionary
    current_file_results = {
        'Document': fname, 
        #'Total_Words': total_words,
        'Abs_mean': abs_mean,
        'Coverage': coverage,
        'Percent_Words_Removed': percent_removed
    }
    current_file_results.update(base_statistics)
    current_file_results.update(stripes_data)
    
    all_document_results.append(current_file_results)
    
    #Adds data for the heatmap plot
    for score in scores:
        stripe_data.append({"File": fname, "Score": score})

if all_document_results:
    summary_df = pd.DataFrame(all_document_results)
    
    #Percentage of words removed after applying the threshold
    summary_df['Percent_Words_Removed'] = summary_df['Percent_Words_Removed']
    
    print("\n\n Analysis Summary: ")
    print(summary_df.head().to_string())
    
    try:
        summary_df.to_excel(output_excel_file, index=False)
        print(f"\n Successfully saved all metrics to: {output_excel_file}")
    except Exception as e:
        print(f"\n ERROR: Could not save the Excel file, due to {e}!")

    #Heatmap for the stripes
    def plot_stripe_percentages(data_list):
        if not data_list: return
        df = pd.DataFrame(data_list)
        bins_plot = np.linspace(-0.5, 0.5, 41)
        labels_plot = [f"[{bins_plot[i]:.2f}, {bins_plot[i+1]:.2f})" for i in range(len(bins_plot)-1)]
        df['bins'] = pd.cut(df['Score'], bins=bins_plot, labels=labels_plot, include_lowest=True)
        bins_counts = df.groupby(['File', 'bins'], observed=False).size().unstack(fill_value=0)
        totals = bins_counts.sum(axis=1)
        bins_percentages = bins_counts.div(totals.replace(0, 1), axis=0) * 100
        heatmap_data = bins_percentages.T.iloc[::-1]
        
        plt.figure(figsize=(12, 8))
        sns.heatmap(heatmap_data, cmap="Blues", annot=True, fmt=".1f", cbar_kws={'label': 'Percentage (%)'}, linewidths=.5)
        plt.title("Percentage of Words per Stripe", fontsize=14)
        plt.ylabel("Stripe Range", fontsize=12)
        plt.xlabel("Document", fontsize=12)
        plt.xticks(rotation=45, ha='right')
        plt.tight_layout()
        plt.show()

    print("\nGenerating heatmap of stripe percentages: ")
    plot_stripe_percentages(stripe_data)

else:
    print("\n Analysis complete, but no data was generated to save!")

In [ ]:
#path to the Excel file with all features
EXCEL_FILE_PATH =r"Result/Filename.xlsx"

#initial alpha parameter
LASSO_ALPHA = 0.1 #the value of alpha needs optimization; for our experiment the range is [0.01 - 0.1]

print(f"Loading data from '{EXCEL_FILE_PATH}'...")
try:
    df = pd.read_excel(EXCEL_FILE_PATH)
except FileNotFoundError:
    print(f"\n The file was not found!")
    exit()

if 'Annotation' not in df.columns:
    print(f"\n 'Annotation' column not found in the Excel file!")
    exit()

print("Data loaded successfully!")

# Defines Features (X) and Target (y); 'y' is the target column we want to predict
y = df['Annotation']

# The features 'X' are all the other numerical columns used as predictors, so all the non-numerical columns are being dropped
X = df.drop(columns=['Document', 'Annotation'])
feature_names = X.columns

"""Lasso is sensitive to the scale of features. Features with large values (like Total_Words) would be unfairly penalized 
compared to features with small values (like 'mean'). To avoid this, the StandardScaler scales all features to have a mean of 0 and a standard deviation of 1."""

print("Scaling features....")
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

#Creates and trains the Lasso model
print(f"Training Lasso model with alpha = {LASSO_ALPHA}:")
lasso = Lasso(alpha=LASSO_ALPHA)
lasso.fit(X_scaled, y)

print("Model training complete!")

# Creates a DataFrame to hold the results
results_df = pd.DataFrame({
    'Feature': feature_names,
    'Coefficient': lasso.coef_
})

print("\n\n Selected features using the Lasso Regression: ")

#Features with non-zero coefficients are the ones "selected" by the model
selected_features = results_df[results_df['Coefficient'] != 0].copy()
selected_features['Abs_Coefficient'] = selected_features['Coefficient'].abs()
selected_features = selected_features.sort_values('Abs_Coefficient', ascending=False).drop(columns='Abs_Coefficient')

#Features with zero coefficients were "discarded"
discarded_features_count = (results_df['Coefficient'] == 0).sum()

if not selected_features.empty:
    print("\nThe following features were selected as important predictors:\n")
    print(selected_features.to_string(index=False))
else:
    print("\nNo features were selected.")
    
print(f"\n{discarded_features_count} features were discarded (coefficient set to zero).")


# Visualization
if not selected_features.empty:
    plt.figure(figsize=(12, 8))
    sns.barplot(
        x='Coefficient', 
        y='Feature', 
        data=selected_features, 
        palette='vlag'
    )
    plt.title(f'Feature Importance from Lasso Regression (alpha={LASSO_ALPHA})', fontsize=16)
    plt.xlabel('Coefficient Value (Predictive Power)', fontsize=12)
    plt.ylabel('Feature', fontsize=12)
    plt.grid(axis='x', linestyle='--', alpha=0.6)
    plt.tight_layout()
    plt.show()

In [ ]:
#Binary Logistic Regression Classifier with LOOCV
if selected_features.empty:
    print("No features were selected by Lasso. Cannot build a Logistic Regression model.")
else:
    final_feature_names = selected_features['Feature'].tolist()
    #print(f"\n Using the {len(final_feature_names)} features selected by Lasso for the model:")
    #print(final_feature_names)
    
    X_final = df[final_feature_names]

    y_binary = df['Annotation']
    print(f"\n Using the existing 'Annotation' column as the binary target.")
    print(f"Class distribution: {(y_binary == 1).sum()} Relevant (1) / {(y_binary == 0).sum()} Not Relevant (0)")
    
    #Leave-One-Out Cross-Validation 
    loocv = LeaveOneOut()
    
    #For storing the results from each of the folds
    all_predictions = []
    all_true_labels = []
    all_probabilities = []
    
    print(f"\n Performing LOOCV with {loocv.get_n_splits(X_final)} folds ...")

    # For the Cross-Validation Loop
    for i, (train_index, test_index) in enumerate(loocv.split(X_final)):
        # Gets the training and testing data for each fold
        X_train, X_test = X_final.iloc[train_index], X_final.iloc[test_index]
        y_train, y_test = y_binary.iloc[train_index], y_binary.iloc[test_index]
        
        # Scales the data
        scaler = StandardScaler()
        X_train_scaled = scaler.fit_transform(X_train)
        X_test_scaled = scaler.transform(X_test)
        
        #Trains the Logistic Regression model
        model = LogisticRegression(random_state=42)
        model.fit(X_train_scaled, y_train)
        
        # Makes a prediction on the single held-out test document
        prediction = model.predict(X_test_scaled)
        probability = model.predict_proba(X_test_scaled)[:, 1] # Gets probability of class 1
        
        #Stores the results
        all_predictions.append(prediction[0])
        all_true_labels.append(y_test.iloc[0])
        all_probabilities.append(probability[0])

    print("LOOCV complete.")

    print("\n Overall Binary Logistic Regression Model Evaluation from LOOCV")

    # Confusion Matrix Visualization
    print("\n Confusion Matrix:")
    cm = confusion_matrix(all_true_labels, all_predictions)
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
                xticklabels=['Predicted Not Relevant (0)', 'Predicted Relevant (1)'],
                yticklabels=['Actual Not Relevant (0)', 'Actual Relevant (1)'])
    plt.title('Overall Confusion Matrix from LOOCV', fontsize=16)
    plt.ylabel('Actual Label')
    plt.xlabel('Predicted Label')
    plt.show()

    print("\n Performance Metrics:")
    print(f"\n Accuracy: {accuracy_score(all_true_labels, all_predictions):.2%}")

    print(f"\n Macro F1-Score: {f1_score(all_true_labels, all_predictions, average='macro', zero_division=0):.3f}")

    print(f"\n Precision: {precision_score(all_true_labels, all_predictions, average='macro', zero_division=0):.3f}")

    print(f"\n Recall: {recall_score(all_true_labels, all_predictions, average='macro', zero_division=0):.3f}")
    
    #print(f"\n Mean Absolute Error: {mean_absolute_error(all_true_labels, all_predictions):.4f}")

    # Final Predictions
    print("\n Final Predictions for All Documents:")
    
    predictions_df = pd.DataFrame({
        'Document': df['Document'],
        'Actual_Annotation': all_true_labels,
        'Predicted_Annotation': all_predictions,
        'Confidence_Probability': all_probabilities #
    }).sort_values('Confidence_Probability', ascending=False)

    #predictions_df['Correctly_Classified'] = (predictions_df['Actual_Annotation'] == predictions_df['Predicted_Annotation'])
    
    print("\nSummary of predictions for all documents, ranked by relevance probability:\n")
    print(predictions_df.to_string(index=False))